# GOES-R ABI — catalog explorer (no network)

The [`earthlens.goes`](../../reference/goes/introduction.md) backend fetches raw
**NOAA GOES-R ABI** NetCDF granules from the anonymous `noaa-goes*` AWS buckets.
Before downloading anything, it helps to understand the three axes of a GOES
request — **satellite**, **product**, and **domain** — and the spectral bands
the imager carries.

This notebook is **fully offline**: it only reads the bundled catalog
(`earthlens.goes.Catalog`). By the end you will know which products, domains,
satellites, and ABI channels are available, and how they map to an S3 key.

## Setup

Load the catalog and pandas for tidy tables.

In [ ]:
import pandas as pd

from earthlens.goes import Catalog

cat = Catalog()
cat  # a compact repr: how many products / available datasets

## Products

A GOES request names a **product family** with `dataset=`. Each row maps a
friendly key to its ABI `product_group` (the S3 prefix stem), the processing
level, and the domains it publishes. `band_split` marks the products that store
**one file per ABI channel** (so `variables=` selects which channels to fetch).

In [ ]:
products = pd.DataFrame(
    [
        {
            "dataset": key,
            "product_group": p.product_group,
            "level": p.level,
            "domains": ", ".join(p.domains),
            "band_split": p.band_split,
            "description": p.description.split(" - ")[0],
        }
        for key, p in sorted(cat.datasets.items())
    ]
)
products

## Domains

ABI scans four **domains**. CONUS (`C`) and Full Disk (`F`) each have their own
S3 prefix suffix; the two Mesoscale sub-sectors (`M1`, `M2`) **share** one `…M`
prefix and are told apart by a filename token. Cadence is informational — the
backend lists the actual hour prefix rather than computing an expected count.

In [ ]:
domains = pd.DataFrame(
    [
        {
            "domain": key,
            "name": d.name,
            "cadence_minutes": d.cadence_minutes,
            "prefix_suffix": d.prefix_suffix,
            "subsector": d.subsector or "-",
        }
        for key, d in cat.domains.items()
    ]
)
domains

## Satellites → buckets

`satellite=` accepts an operational **role** (`east` / `west`) or an explicit
satellite **number** (`16` / `18` / `19`). The role→bucket map lives in the
catalog because East/West rotate as new GOES satellites commission.

In [ ]:
pd.DataFrame(sorted(cat.satellites.items()), columns=["satellite", "bucket"])

## ABI spectral channels

The imager carries **16 spectral bands**. For the band-split products
(`abi-l1b-rad`, `abi-l2-cmip`) you pick channels with
`variables=["C02", "C13", ...]`. The visible / near-IR bands sit at short
wavelengths; the water-vapour and window bands are thermal-infrared.

In [ ]:
channels = pd.DataFrame(
    [
        {"channel": k, "wavelength_um": c.wavelength_um, "name": c.name}
        for k, c in cat.channels.items()
    ]
)
channels

A quick look at where each band sits on the spectrum:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(channels["channel"], channels["wavelength_um"], color="#3b7dd8")
ax.set_ylabel("wavelength (µm)")
ax.set_title("ABI spectral channels")
ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

## From a request to an S3 key (still no network)

Constructing the backend resolves the satellite to a bucket and the
product + domain to an S3 prefix — **without touching the network**. This is the
prefix the backend will list per hour of the requested window.

In [ ]:
from earthlens.goes import GOES

goes = GOES(
    start="2026-07-03 12:00",
    end="2026-07-03 12:10",
    lat_lim=[20, 50],
    lon_lim=[-130, -60],
    dataset="abi-l2-mcmip",
    satellite="east",
    domain="C",
    fmt="%Y-%m-%d %H:%M",
)
print("bucket:", goes._bucket)
print("prefix:", goes._prefix())
print("hours :", [str(h) for h in goes.time.dates])

## Takeaway

* A GOES request is **satellite × product × domain × time window**.
* `dataset=` picks the product family; `variables=` picks ABI channels for the
  band-split products; `satellite=` and `domain=` resolve to a bucket + S3
  prefix, all from the catalog.
* Next: the [CONUS granule download](conus_granule_download.ipynb) notebook
  fetches a real granule and decodes a band with pyramids.